# Counting vehicles by class for signal timing (UVH-26, YOLOv8)

AIML ZG525 Computer Vision, EC-1 Assignment 1 · Track B (object detection)

This notebook runs top to bottom on a free Colab T4 (*Runtime → Change runtime type → T4 GPU*).

- **`RUN_TRAINING`** (next cell): `True` retrains both models; `False` skips training and evaluates the weights saved on Drive.
- The dataset is downloaded by the notebook from the Hugging Face Hub at a pinned revision; no data files are embedded.
- Sections follow the report's 8 steps. Every number quoted in the report appears in a cell output below.
- Optional Colab secrets: `HF_TOKEN` (faster Hub downloads), `GITHUB_TOKEN` (only if the repo is private).

In [ ]:
# ---- Run settings: the only cell a grader should need to edit ----
RUN_TRAINING = True   # False: skip training, evaluate the weights saved under DRIVE_ROOT/runs
REPO_URL = "https://github.com/YashRenwa/uvh26-pcu-counting.git"
COMMIT = "main"       # pinned to a commit hash for the final submission run
DRIVE_ROOT = "/content/drive/MyDrive/uvh26"

In [ ]:
import os, subprocess, sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules


def sh(cmd):
    print("$", cmd)
    subprocess.run(cmd, shell=True, check=True)


if IN_COLAB:
    from google.colab import drive, userdata
    drive.mount("/content/drive")
    repo_dir = Path("/content/uvh26-pcu-counting")
    if not repo_dir.exists():
        try:
            token = userdata.get("GITHUB_TOKEN")  # only needed for a private repo
        except Exception:
            token = None
        url = REPO_URL.replace("https://", f"https://{token}@") if token else REPO_URL
        subprocess.run(f"git clone -q {url} {repo_dir}", shell=True, check=True)
    os.chdir(repo_dir)
    sh(f"git fetch -q && git checkout -q {COMMIT}")
    # Keep Colab's preinstalled torch/opencv/numpy (CUDA-matched, no restart); pin the rest.
    sh("grep -v colab-skip requirements.txt > /tmp/req-colab.txt && pip install -q -r /tmp/req-colab.txt")
    os.environ.update({
        "UVH_RAW_ROOT": "/content/raw",              # original PNGs on fast local disk
        "UVH_DATA_ROOT": f"{DRIVE_ROOT}/prepared",   # JPEGs + labels survive a disconnect
        "UVH_RUNS_ROOT": f"{DRIVE_ROOT}/runs",
        "UVH_WEIGHTS_DIR": f"{DRIVE_ROOT}/weights",
    })
else:
    repo_dir = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
    os.chdir(repo_dir)

sys.path.insert(0, str(repo_dir))
commit = subprocess.run("git rev-parse --short HEAD", shell=True, capture_output=True, text=True).stdout.strip()
print("repo:", repo_dir, "| commit:", commit or "(no commits yet)")

In [ ]:
import torch
from src.common import library_versions, load_config, set_seed

cfg = load_config()
set_seed(cfg["seed"])
print("seed:", cfg["seed"])
for name, version in library_versions().items():
    print(f"  {name:16s} {version}")
print("device:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu")
print("paths:", {k: str(v) for k, v in cfg["paths"].items()})

## 1. Problem and task type

*Written with the report.* Signal timing needs per-class vehicle **counts** at a junction, converted to PCU. Counting needs instances, so this is object detection: classification cannot count, and segmentation adds labelling and compute cost without changing the count.

## 2. Data and split  *(Phase 1)*

**Source.** UVH-26 (IISc AIM with the Bengaluru Traffic Police), **CC BY 4.0**, downloaded from the Hugging Face Hub at the revision pinned in `configs/base.yaml`. The full set is 26,646 PNGs (90 GB), so only the four annotation JSONs and a seeded pool of images are fetched.

**Pool.** STAPLE labels exist for only 81% of images (MV covers all), so the pool is drawn from images that carry both label sets. The MV and STAPLE runs then train on identical images. Sampling keeps the official train/val proportion; `splits/pool.csv` records exactly which images were used.

**Why not the official split.** UVH-26 has no camera ids, and a fixed CCTV camera produces many near-identical frames. We group images into *pseudo-cameras* (CLIP ViT-B/32 embeddings, cosine similarity ≥ τ, connected components), measure how often the official split puts one camera on both sides, and then split whole pseudo-cameras 70/15/15.

**Frozen split.** The split was built once and committed (`splits/groups.csv`, `splits/{train,val,test}.txt`, `splits/test.sha256`). CLIP outputs differ slightly between CPU and GPU, so this notebook recomputes the grouping to show the evidence, but trains and evaluates on the committed split after verifying the test-set checksum.

In [ ]:
import pandas as pd
from IPython.display import Image, display
from src import cluster, download, prepare, split

FIGS = cfg["paths"]["results_dir"] / "figures"
pool = download.select_pool(cfg)   # reads splits/pool.csv from the repo
print(f"pool: {len(pool)} images with both MV and STAPLE labels")
print(pool.groupby(["official_split", "subfolder"]).size().rename("images").to_string())
download.download_pool(cfg, pool)  # PNGs to local disk; skips images already prepared on Drive

Resize to 1280 px on the long side (JPEG quality 95) and write YOLO labels twice, once from MV and once from STAPLE. YOLO box coordinates are normalised, so they need no rescaling.

In [ ]:
prep = prepare.run(cfg, pool)
print({k: v for k, v in prep.items() if k not in ("class_counts", "subfolders")})
display(pd.DataFrame(prep["class_counts"]).rename(columns=str.upper))
display(Image(filename=str(prepare.draw_labels(cfg, pool["stem"].iloc[0], FIGS / "labels_mv_vs_st.jpg")), width=900))

### Pseudo-cameras and the official split's leakage

In [ ]:
clu = cluster.run(cfg)
pc, lk = clu["pseudo_cameras"], clu["leakage"]
print(f"tau = {clu['tau']} | {pc['groups']} pseudo-cameras | {pc['singletons']} singletons | largest {pc['largest_group']} images")
print(f"LEAKAGE: {lk['val_images_sharing_camera_with_train']} of {lk['val_images']} official-val images "
      f"({lk['leakage_val_share']:.1%}) share a pseudo-camera with an official-train image")
if clu["frozen_groups_used"]:
    print("frozen groups used; recomputed grouping identical:", clu["recomputed_groups_identical"])
display(pd.DataFrame(clu["tau_sweep"]).set_index("tau"))

In [ ]:
for name in ("nn_similarity.png", "group_sizes.png"):
    display(Image(filename=str(FIGS / name), width=640))
for name in ("pairs_above_tau.jpg", "pairs_below_tau.jpg", "pairs_weakest_in_group.jpg", "leak_examples.jpg"):
    display(Image(filename=str(FIGS / name), width=900))

### Group split and class balance

In [ ]:
if (cfg["paths"]["splits_dir"] / "test.sha256").exists():
    print("split is frozen; test.txt sha256 verified:", split.check_test_checksum(cfg))
    assignment = split.load(cfg)
else:
    assignment = split.make(cfg)
spl = split.summarise(cfg, assignment)
floor = cfg["split"]["min_test_instances"]
for k in ("images", "image_share", "pseudo_cameras", "official_split_composition",
          "classes_missing", f"classes_under_{floor}_test_instances"):
    print(f"{k}: {spl[k]}")
display(pd.read_csv(cfg["paths"]["results_dir"] / "split_class_counts.csv", header=[0, 1], index_col=0))

## 3. Preprocessing and augmentation  *(Phases 1-2)*

## 4. Baseline: YOLOv8 fine-tuned on MV labels  *(Phase 2)*

## 5. Ablation: MV vs STAPLE training labels  *(Phase 3)*

## 6. Evaluation  *(Phase 4 threshold on val; Phase 5 sealed test, once)*

## 7. Error analysis  *(Phase 6)*

## 8. Conclusion